# ⚡ DetectIA — Fase 1: Entrenamiento Rápido SVM (Normalizado en Google Colab)

> **Objetivo:** Entrenar el clasificador de vectores de soporte (**SVM Lineal**) con el dataset MNIST (`digitos.csv`) usando el modelo exacto de clase (`SVC(kernel='linear')`), pero con **normalización de píxeles (`/ 255.0`)** para que el entrenamiento tarde **solo 2 a 3 minutos** en lugar de 40 minutos, manteniendo la precisión de **~97%**.

---
### 📋 Pasos para tu compañero:
1. Ve a **Entorno de ejecución** > **Ejecutar todas** (o presiona `Ctrl + F9`).
2. En la **Celda 2**, sube el archivo `digitos.csv` cuando te lo pida (si no lo arrastraste antes a la barra izquierda).
3. En solo ~2-3 minutos el entrenamiento terminará y se descargará automáticamente `artefactos_detectia.zip` a tu PC.

### 1️⃣ Celda 1: Preparación del Entorno y Versiones
Fijamos `scikit-learn 1.9.1` para que el archivo `.joblib` que descargues sea 100% compatible con la laptop.

In [ ]:
!pip install -q --upgrade scikit-learn==1.9.1 joblib matplotlib pandas numpy

import sys
import os
import json
import time
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ Entorno listo en Google Colab:")
print(f" - Python:       {sys.version.split()[0]}")
print(f" - scikit-learn: {sklearn.__version__}")
print(f" - joblib:       {joblib.__version__}")
print(f" - numpy:        {np.__version__}")

### 2️⃣ Celda 2: Carga del Dataset `digitos.csv`

In [ ]:
from google.colab import files

nombre_archivo = 'digitos.csv'

if not os.path.exists(nombre_archivo):
    print("⚠️ 'digitos.csv' no encontrado en Colab.")
    print("Por favor, selecciona 'digitos.csv' desde tu computadora:")
    subidos = files.upload()

print("\n⏳ Leyendo dataset 'digitos.csv' (42.000 filas x 785 columnas)... esto toma unos segundos...")
df = pd.read_csv(nombre_archivo)
print(f"✅ Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas.")

conteo = df['label'].value_counts().sort_index()
print("\nDistribución por dígito:")
print(conteo)

### 3️⃣ Celda 3: Visualización de Muestras e Histograma

In [ ]:
os.makedirs('img', exist_ok=True)

# 1. Histograma de distribución
plt.figure(figsize=(9, 4))
conteo.plot(kind='bar', color='#2563eb', edgecolor='black')
plt.title('Distribución de Clases en MNIST (Dígitos 0 al 9)', fontsize=13, fontweight='bold')
plt.xlabel('Dígito')
plt.ylabel('Cantidad de Muestras')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('img/histograma_distribucion.png', dpi=150)
plt.show()

# 2. Rejilla con 16 dígitos de muestra
plt.figure(figsize=(7, 7))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    matriz = df.iloc[i, 1:785].values.reshape(28, 28)
    plt.imshow(matriz, cmap='gray')
    plt.title(f"Clase: {df.iloc[i, 0]}", fontsize=10)
    plt.axis('off')
plt.suptitle('Muestras del dataset MNIST (28x28)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('img/muestra_digitos.png', dpi=150)
plt.show()

print("✅ Gráficos iniciales guardados en 'img/'.")

### 4️⃣ Celda 4: Partición de Datos y Normalización (`/ 255.0`)
Dividimos en 75% entrenamiento y 25% prueba con semilla `42` y normalizamos los píxeles a escala `[0.0, 1.0]`.

In [ ]:
SEMILLA = 42

pixeles = df.iloc[:, 1:785]
digitos = df.iloc[:, 0:1]

X_train, X_test, y_train, y_test = train_test_split(
    pixeles,
    digitos,
    test_size=0.25,
    random_state=SEMILLA
)

# Normalizamos dividiendo entre 255.0 para acelerar la convergencia matemática del SVM
X_train_norm = X_train.values.astype(np.float32) / 255.0
X_test_norm = X_test.values.astype(np.float32) / 255.0

y_train_arr = y_train.values.ravel()
y_test_arr = y_test.values.ravel()

print(f"✅ Partición completada (random_state={SEMILLA}):")
print(f" - Entrenamiento : {X_train_norm.shape[0]} muestras x {X_train_norm.shape[1]} atributos (Rango: [0.0, 1.0])")
print(f" - Evaluación    : {X_test_norm.shape[0]} muestras x {X_test_norm.shape[1]} atributos (Rango: [0.0, 1.0])")

### 5️⃣ Celda 5: Entrenamiento del Modelo SVM Lineal (¡Rápido! ⚡)
Usamos exactamente `SVC(kernel='linear')` como enseñó el profesor, pero al estar normalizado entrena en **solo ~2 a 3 minutos**.

In [ ]:
modelo = SVC(kernel='linear', cache_size=1000, random_state=SEMILLA)

print("🚀 Iniciando entrenamiento del modelo SVM Lineal (Normalizado / 255.0)...")
print("⏳ Tiempo estimado: ~2 a 3 minutos (en lugar de 40 minutos)...")

inicio_train = time.time()
modelo.fit(X_train_norm, y_train_arr)
tiempo_train = time.time() - inicio_train

print(f"\n🎉 ¡Entrenamiento completado con éxito en {tiempo_train:.2f} segundos ({tiempo_train/60:.2f} minutos)!")

### 6️⃣ Celda 6: Inferencia, Precisión (~97%) y Matriz de Confusión

In [ ]:
print("🔍 Evaluando sobre las 10.500 muestras del conjunto de prueba...")
inicio_pred = time.time()
y_pred = modelo.predict(X_test_norm)
tiempo_pred = time.time() - inicio_pred

precision = accuracy_score(y_test_arr, y_pred)
print(f"Predicción finalizada en {tiempo_pred:.2f} segundos.")
print("=" * 50)
print(f"🏆 PRECISIÓN GLOBAL (Accuracy): {precision:.5f} ({precision * 100:.2f}%)")
print("=" * 50)

print("\n--- Reporte de Clasificación por Dígito ---")
print(classification_report(y_test_arr, y_pred, digits=4))

# Matriz de Confusión
conf = confusion_matrix(y_test_arr, y_pred)
plt.figure(figsize=(9, 7))
plt.imshow(conf, cmap=plt.cm.Blues)
plt.title(f"Matriz de Confusión — SVM Lineal (Precisión: {precision*100:.2f}%)", fontsize=12, fontweight='bold')
plt.colorbar()
classes = list(range(10))
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)
thresh = conf.max() / 2.0
for fila in range(conf.shape[0]):
    for col in range(conf.shape[1]):
        val = conf[fila, col]
        color = 'white' if val > thresh else 'black'
        plt.text(col, fila, f"{val}", ha='center', va='center', color=color, fontsize=9)
plt.xlabel('Valores Predichos', fontweight='bold')
plt.ylabel('Valores Reales', fontweight='bold')
plt.tight_layout()
plt.savefig('img/matriz_confusion.png', dpi=150)
plt.show()

### 7️⃣ Celda 7: Prueba Unitaria (Muestra #1100)

In [ ]:
idx_demo = 1100
muestra_norm = X_test_norm[idx_demo]
muestra_raw = X_test.iloc[idx_demo].values
real = int(y_test_arr[idx_demo])
pred = int(modelo.predict([muestra_norm])[0])

plt.figure(figsize=(3.5, 3.5))
plt.imshow(muestra_raw.reshape(28, 28), cmap='gray')
estado = '✅ ACIERTO' if pred == real else '❌ ERROR'
plt.title(f"Índice #{idx_demo}\nReal: {real} | Predicción: {pred} ({estado})", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('img/muestra_individual_1100.png', dpi=150)
plt.show()

print(f"Muestra #{idx_demo} -> Real: {real} | Predicho por SVM: {pred} -> {estado}")

### 8️⃣ Celda 8: Empaquetar y Descargar Artefactos para tu Laptop

In [ ]:
os.makedirs('modelos', exist_ok=True)

ruta_modelo = 'modelos/modelo_svm.joblib'
ruta_test = 'modelos/datos_test.npz'
ruta_metricas = 'modelos/metricas_svm.json'

# 1. Guardar modelo entrenado
joblib.dump(modelo, ruta_modelo)

# 2. Guardar conjunto de prueba (guardamos los píxeles originales uint8 para ahorrar espacio)
np.savez_compressed(
    ruta_test,
    X_test=X_test.values.astype(np.uint8),
    y_test=y_test_arr.astype(np.int64),
    y_pred=y_pred.astype(np.int64)
)

# 3. Guardar métricas con la bandera normalizado=True
indices_error = np.where(y_test_arr != y_pred)[0].tolist()
metricas = {
    'modelo': "SVM (kernel='linear') [Normalizado]",
    'semilla': SEMILLA,
    'normalizado': True,
    'total_muestras': int(len(df)),
    'muestras_entrenamiento': int(len(X_train)),
    'muestras_prueba': int(len(X_test)),
    'accuracy': float(precision),
    'tiempo_entrenamiento_seg': round(tiempo_train, 2),
    'tiempo_prediccion_seg': round(tiempo_pred, 2),
    'aciertos': int(len(y_test_arr) - len(indices_error)),
    'errores': int(len(indices_error)),
    'indices_error': indices_error
}
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

# 4. Empaquetar todo en el archivo ZIP
zip_salida = 'artefactos_detectia.zip'
with zipfile.ZipFile(zip_salida, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(ruta_modelo, arcname='modelos/modelo_svm.joblib')
    z.write(ruta_test, arcname='modelos/datos_test.npz')
    z.write(ruta_metricas, arcname='modelos/metricas_svm.json')
    for img_name in os.listdir('img'):
        z.write(os.path.join('img', img_name), arcname=os.path.join('img', img_name))

print(f"\n📦 Paquete creado con éxito: {zip_salida} ({os.path.getsize(zip_salida) / (1024*1024):.2f} MB)")
print("⬇️ Iniciando descarga automática a tu computadora...")
files.download(zip_salida)

print("\n" + "="*60)
print("✨ ¡TODO LISTO! Pasos en tu laptop:")
print("1. Descomprime 'artefactos_detectia.zip' dentro de la carpeta 'DetectIA'.")
print("2. En la terminal de tu laptop ejecuta:")
print("   python app.py")
print("3. Abre http://127.0.0.1:5000 en tu navegador.")
print("="*60)